# Rattrapage - alignement rapport 14.06.26

Ce notebook est volontairement leger. Il orchestre le code officiel du projet dans `src` au lieu de recopier les fonctions d'entrainement, d'evaluation ou de Grad-CAM.

Objectifs :
- verifier que VGG16, EfficientNetB0, ResNet50 et ResNet50 masque sont alignes avec le rapport ;
- lancer uniquement les etapes manquantes si besoin ;
- lire les metriques et figures deja produites dans `reports`.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / "src").exists():
    candidates = [p for p in PROJECT_DIR.parents if (p / "src").exists()]
    if not candidates:
        raise RuntimeError("Impossible de trouver la racine du projet contenant src/.")
    PROJECT_DIR = candidates[0]

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

os.environ.setdefault("MPLCONFIGDIR", "/tmp")

from src.utils.env import ensure_dirs, get_paths, setup_environment
from src.models import transfer_learning as tl

info = setup_environment()
paths = ensure_dirs()

print("Racine projet :", PROJECT_DIR)
print("Backbones disponibles :", sorted(tl.BACKBONES))


## 1. Verification de l'alignement des backbones

Le rapport de reference mentionne VGG16, EfficientNetB0, ResNet50 et l'experience corrective ResNet50 avec masquage pulmonaire. Cette cellule verifie que les trois backbones non masques sont bien declares dans l'architecture commune.


In [ ]:
expected = ["vgg16", "efficientnetb0", "resnet50"]
rows = []
for name in expected:
    spec = tl.get_spec(name)
    rows.append({
        "backbone": name,
        "img_size": spec.img_size,
        "last_conv_layer": spec.last_conv_layer,
        "finetune_from": ", ".join(spec.finetune_from_prefixes),
    })

pd.DataFrame(rows)


## 2. Etat des artefacts attendus

Cette cellule ne lance aucun entrainement. Elle signale simplement les fichiers que le rapport exploite ou que les commandes suivantes peuvent regenerer.


In [ ]:
expected_artifacts = {
    "metrics_resnet50": paths["metrics"] / "resnet50_test_metrics.json",
    "metrics_resnet50_masked": paths["metrics"] / "resnet50_masked_test_metrics.json",
    "bias_resnet50_masked": paths["metrics"] / "resnet50_masked_bias_summary.json",
    "metrics_efficientnetb0": paths["metrics"] / "efficientnetb0_test_metrics.json",
    "metrics_vgg16": paths["metrics"] / "vgg16_test_metrics.json",
    "model_resnet50_masked": paths["models_transfer"] / "resnet50_masked_best.keras",
}

artifact_rows = [
    {"artefact": key, "path": str(path), "exists": path.exists()}
    for key, path in expected_artifacts.items()
]
pd.DataFrame(artifact_rows)


## 3. Entrainement VGG16 si les metriques manquent

Par defaut, la cellule affiche la commande sans l'executer. Passer `RUN_VGG16 = True` pour lancer l'entrainement VGG16 officiel via `src.models.run_transfer_learning`.

Le runner applique le protocole du rapport pour VGG16 : 3 epoques de tete avec `lr=1e-4`, puis 3 epoques de fine-tuning avec `lr=1e-5`.


In [ ]:
RUN_VGG16 = False
cmd = [sys.executable, "-m", "src.models.run_transfer_learning", "vgg16"]

if RUN_VGG16:
    env = {**os.environ, "PYTHONPATH": str(PROJECT_DIR), "MPLCONFIGDIR": "/tmp"}
    subprocess.run(cmd, check=True, env=env)
else:
    print("Commande a lancer si besoin :")
    print("PYTHONPATH=. MPLCONFIGDIR=/tmp python -m src.models.run_transfer_learning vgg16")


## 4. Rattrapage ResNet50 masque

Cette experience applique les masques pulmonaires pendant l'entrainement et l'evaluation. Elle sert a verifier la robustesse clinique de l'interpretation Grad-CAM, pas a remplacer automatiquement le meilleur modele quantitatif.


In [ ]:
RUN_MASKED_RESNET50 = False
cmd = [
    sys.executable, "-m", "src.models.train_masked",
    "resnet50", "--batch-size", "16", "--gradcam-n", "12",
]

if RUN_MASKED_RESNET50:
    env = {**os.environ, "PYTHONPATH": str(PROJECT_DIR), "MPLCONFIGDIR": "/tmp"}
    subprocess.run(cmd, check=True, env=env)
else:
    print("Commande a lancer si besoin :")
    print("PYTHONPATH=. MPLCONFIGDIR=/tmp python -m src.models.train_masked resnet50 --batch-size 16 --gradcam-n 12")


## 5. Comparaison ResNet50 initial vs ResNet50 masque

La comparaison est calculee depuis les JSON de `reports/metrics`, sur les memes splits train/validation/test figes.


In [ ]:
metrics_dir = paths["metrics"]
baseline_path = metrics_dir / "resnet50_test_metrics.json"
masked_path = metrics_dir / "resnet50_masked_test_metrics.json"
bias_path = metrics_dir / "resnet50_masked_bias_summary.json"

missing = [p for p in [baseline_path, masked_path] if not p.exists()]
if missing:
    print("Metriques manquantes :")
    for path in missing:
        print("-", path)
else:
    baseline = json.loads(baseline_path.read_text(encoding="utf-8"))
    masked = json.loads(masked_path.read_text(encoding="utf-8"))
    bias = json.loads(bias_path.read_text(encoding="utf-8")) if bias_path.exists() else {}

    comparison = pd.DataFrame([
        {
            "modele": "ResNet50 initial",
            "accuracy": baseline["accuracy"],
            "balanced_accuracy": baseline["balanced_accuracy"],
            "f1_macro": baseline["f1_macro"],
            "recall_COVID": baseline["per_class"]["COVID"]["recall"],
            "gradcam_dans_poumons": None,
        },
        {
            "modele": "ResNet50 masque",
            "accuracy": masked["accuracy"],
            "balanced_accuracy": masked["balanced_accuracy"],
            "f1_macro": masked["f1_macro"],
            "recall_COVID": masked["per_class"]["COVID"]["recall"],
            "gradcam_dans_poumons": bias.get("gradcam_in_lungs"),
        },
    ])
    display(comparison)

    if bias:
        print(
            "Grad-CAM dans les poumons apres masquage : "
            f"{100 * bias['gradcam_in_lungs']:.1f}% "
            f"(surface pulmonaire moyenne : {100 * bias['lung_area_reference']:.1f}%)"
        )


## 6. Figures Grad-CAM du ResNet50 masque

Affiche les panneaux generes par `src.models.train_masked`.


In [ ]:
figures = sorted(paths["figures"].glob("gradcam_resnet50_masked_COVID_*.png"))
if not figures:
    print("Aucune figure Grad-CAM masquee trouvee dans", paths["figures"])
else:
    for fig_path in figures:
        print(fig_path)
        display(Image(filename=str(fig_path)))


## 7. Conclusion operationnelle

- Si `vgg16_test_metrics.json` manque, lancer la cellule VGG16.
- Si les artefacts `resnet50_masked_*` manquent, lancer la cellule ResNet50 masque.
- Les commentaires du rapport restent compatibles avec l'architecture tant que les metriques JSON et les figures ci-dessus sont presentes.
